<a href="https://colab.research.google.com/github/sittidetw/DADS6003/blob/main/hw4_classification/DADS6003_HW4_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4: Workshop  
<b>Instructions</b>  
Using the provided dataset (train.csv and test.csv) on credit score ratings, create a Google Colab notebook to develop your best classification model. You must use only Logistic Regression and/or Naive Bayes to classify customers' credit scores from 'train.csv' into three categories: 'Standard,' 'Good,' and 'Poor' (within the 'Credit_Score' column). Your model will be evaluated using the F1 score.  

Please submit a shareable link to your notebook, along with a 'result.csv' file. This file should contain the original 'test.csv' data with your predicted classifications added to the 'Credit_Score' column.

In [1]:
!git clone https://github.com/sittidetw/DADS6003.git

Cloning into 'DADS6003'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 38 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 9.93 MiB | 4.46 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [3]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Load Train dataset
file_path_train = '/content/DADS6003/hw4_classification/raw_data/train.csv'
try:
    df_train = pd.read_csv(file_path_train)
    print("Data loaded successfully. Shape:", df_train.shape)
    display(df_train.info())
except FileNotFoundError:
    print(f"File not found at {file_path_train}. Please upload the file to colab.")

Data loaded successfully. Shape: (80000, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        80000 non-null  object 
 1   Customer_ID               80000 non-null  object 
 2   Month                     80000 non-null  object 
 3   Name                      71992 non-null  object 
 4   Age                       80000 non-null  object 
 5   SSN                       80000 non-null  object 
 6   Occupation                80000 non-null  object 
 7   Annual_Income             80000 non-null  object 
 8   Monthly_Inhand_Salary     68038 non-null  float64
 9   Num_Bank_Accounts         80000 non-null  int64  
 10  Num_Credit_Card           80000 non-null  int64  
 11  Interest_Rate             80000 non-null  int64  
 12  Num_of_Loan               80000 non-null  object 
 13  Type_of_Loan    

None

In [6]:
# Load Test Dataset
file_path_test = '/content/DADS6003/hw4_classification/raw_data/test.csv'
try:
    df_test = pd.read_csv(file_path_test)
    print("Data loaded successfully. Shape:", df_test.shape)
    display(df_test.info())
except FileNotFoundError:
    print(f"File not found at {file_path_test}. Please upload the file to colab.")

Data loaded successfully. Shape: (20000, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        20000 non-null  object 
 1   Customer_ID               20000 non-null  object 
 2   Month                     20000 non-null  object 
 3   Name                      18023 non-null  object 
 4   Age                       20000 non-null  object 
 5   SSN                       20000 non-null  object 
 6   Occupation                20000 non-null  object 
 7   Annual_Income             20000 non-null  object 
 8   Monthly_Inhand_Salary     16960 non-null  float64
 9   Num_Bank_Accounts         20000 non-null  int64  
 10  Num_Credit_Card           20000 non-null  int64  
 11  Interest_Rate             20000 non-null  int64  
 12  Num_of_Loan               20000 non-null  object 
 13  Type_of_Loan    

None

## Data Cleaning

In [11]:
# ฟังก์ชันสำหรับทำความสะอาดคอลัมน์ตัวเลขที่มีอักขระพิเศษ (เช่น '_', สัญลักษณ์ต่างๆ)
def clean_numeric_string(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    # เก็บเฉพาะตัวเลข จุดทศนิยม และเครื่องหมายลบ
    cleaned_val = re.sub(r'[^\d.-]', '', val)
    try:
        return float(cleaned_val) if cleaned_val else np.nan
    except ValueError:
        return np.nan

# แปลงคอลัมน์ Credit_History_Age ("XX Years and YY Months" -> จำนวนเดือนรวม)
def convert_age_to_months(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    years = re.search(r'(\d+) Years', val)
    months = re.search(r'(\d+) Months', val)

    y = int(years.group(1)) if years else 0
    m = int(months.group(1)) if months else 0
    return (y * 12) + m

In [12]:
def data_cleaning(df_temp, global_medians = None):
  df = df_temp.copy()

  # Drop unused columns
  columns_to_drop = ['Month', 'Name', 'SSN', 'Occupation','Type_of_Loan', 'Credit_Mix', 'Payment_Behaviour']
  df.drop(columns=columns_to_drop, inplace=True)
  df.set_index('ID', inplace=True)

  # Clean columns with messy numbers
  messy_num_cols = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
    'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
    'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance'
  ]
  for col in messy_num_cols:
    df[col] = df[col].apply(clean_numeric_string)

  # Convert Credit_History_Age to Months
  df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_age_to_months)

  # Select Columns
  numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
  categorical_cols = df.select_dtypes(include=['object']).columns.drop('Credit_Score') # ไม่ยุ่งกับ Target

  # Handling missing values by grouping Customer_ID
  group_means = df.groupby('Customer_ID')[numeric_cols].transform('mean')
  df[numeric_cols] = df[numeric_cols].fillna(group_means)

  # Fallback: using global medians
  if global_medians == None: global_medians = df[numeric_cols].median()
  df[numeric_cols] = df[numeric_cols].fillna(global_medians)

  # Handling missing values for categorical data
  # print(f'Before: ',list(df['Occupation'].unique()))
  # df['Occupation'] = df['Occupation'].replace('_______', np.nan)
  # df['Occupation'] = df.groupby('Customer_ID')['Occupation'].transform(lambda x: x.ffill().bfill())
  # print(f'After: ',list(df['Occupation'].unique()))

  # One-Hot Encoding
  df = pd.get_dummies(df, columns=['Payment_of_Min_Amount'], drop_first=True)

  # Sort Columns
  cols = [col for col in df.columns if col != 'Credit_Score']
  cols.append('Credit_Score')
  df = df[cols]

  df.drop(columns=['Customer_ID'], inplace=True)
  return df, global_medians

df, global_medians = data_cleaning(df_train)
display(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 80000 entries, 0x1d1af to 0x2468
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        80000 non-null  float64
 1   Annual_Income              80000 non-null  float64
 2   Monthly_Inhand_Salary      80000 non-null  float64
 3   Num_Bank_Accounts          80000 non-null  float64
 4   Num_Credit_Card            80000 non-null  float64
 5   Interest_Rate              80000 non-null  float64
 6   Num_of_Loan                80000 non-null  float64
 7   Delay_from_due_date        80000 non-null  float64
 8   Num_of_Delayed_Payment     80000 non-null  float64
 9   Changed_Credit_Limit       80000 non-null  float64
 10  Num_Credit_Inquiries       80000 non-null  float64
 11  Outstanding_Debt           80000 non-null  float64
 12  Credit_Utilization_Ratio   80000 non-null  float64
 13  Credit_History_Age         80000 non-null  f

None

In [13]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Age,80000.0,1.101239e+02,6.810537e+02,-5.000000e+02,24.000000,33.000000,42.000000,8.698000e+03
Annual_Income,80000.0,1.777297e+05,1.428177e+06,7.005930e+03,19433.067500,37550.740000,72896.120000,2.417715e+07
Monthly_Inhand_Salary,80000.0,4.201942e+03,3.199703e+03,3.036454e+02,1623.664167,3086.756667,5964.883333,1.520463e+04
Num_Bank_Accounts,80000.0,1.673690e+01,1.155007e+02,-1.000000e+00,3.000000,6.000000,7.000000,1.798000e+03
Num_Credit_Card,80000.0,2.248903e+01,1.292278e+02,0.000000e+00,4.000000,5.000000,7.000000,1.499000e+03
Interest_Rate,80000.0,7.008293e+01,4.561224e+02,1.000000e+00,8.000000,13.000000,20.000000,5.789000e+03
Num_of_Loan,80000.0,3.141250e+00,6.345849e+01,-1.000000e+02,1.000000,3.000000,5.000000,1.496000e+03
Delay_from_due_date,80000.0,2.106798e+01,1.487683e+01,-5.000000e+00,10.000000,18.000000,28.000000,6.700000e+01
Num_of_Delayed_Payment,80000.0,3.123922e+01,2.213484e+02,-3.000000e+00,9.000000,14.000000,18.000000,4.397000e+03
Changed_Credit_Limit,80000.0,1.038316e+01,6.785893e+00,-6.480000e+00,5.330000,9.400000,14.810000,3.697000e+01
